In [ ]:
import pandas as pd
labels = pd.read_csv("../data/raw/preprocessed/anomaly_label.csv")
labels.head()

In [ ]:
import re
from collections import Counter
from pathlib import Path
import pandas as pd

SAMPLE = Path("../tests/data/HDFS_2k.log")
lines = SAMPLE.read_text(encoding="utf-8", errors="replace").splitlines()
print(len(lines))
print(*lines[:8], sep="\n")

In [ ]:
# Découpage naïf : Date Heure PID Niveau Composant: Message
cols = ["date", "time", "pid", "level", "component", "content"]
parts = [l.split(" ", 5) for l in lines]
print("lignes à 6 champs :", sum(len(p) == 6 for p in parts), "/", len(lines))
df = pd.DataFrame([p for p in parts if len(p) == 6], columns=cols)

print(df["level"].value_counts())
print(df["component"].value_counts().head(10))
print("période :", df["date"].min(), "→", df["date"].max())

In [ ]:
# Identifiants de blocs
blocks = [re.findall(r"blk_-?\d+", l) for l in lines]
print("nombre de blocs par ligne :")
print(pd.Series([len(b) for b in blocks]).value_counts().sort_index())

print("\nexemples de lignes à plusieurs blocs :")
print(*[l for l, b in zip(lines, blocks) if len(b) > 1][:3], sep="\n")

per_block = Counter(b for bl in blocks for b in bl)
print("\nblocs distincts :", len(per_block))
print("lignes par bloc :", pd.Series(per_block).describe().round(1).to_dict())

In [ ]:
# Pourquoi il faut un vrai parseur : variété des messages
contents = df["content"]
naive = contents.str.replace(r"\d+", "N", regex=True)
print("messages distincts          :", contents.nunique())
print("après remplacement des nombres :", naive.nunique())